# Construction of Daily Geopolitical Risk Indices

<p><i>Author:</i> Angelica Vanti</p>

<p><i>Project:</i> Predicting EUR/USD Movements Using Geopolitical and Macroeconomic Variables</p>

This notebook transforms the final tweet-level LLM geopolitical-risk scores into daily time-series indices for subsequent econometric modelling.

Three geopolitical dimensions are considered: **Trade Hostility**, **Sanctions Threat**, and **Federal Reserve Pressure**. Alternative daily aggregation methods are constructed to examine whether different representations of geopolitical information provide greater predictive value for EUR/USD movements.

The notebook produces daily maximum and daily summed scores, together with 3-, 5-, and 10-day moving averages of the summed indices.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Repository paths
ROOT = Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"

# Input: final tweet-level geopolitical annotations
ANNOTATIONS_PATH = (
    PROCESSED_DIR
    / "LLM_annotations_max700_6examples.csv"
)

# Outputs
DAILY_MAX_PATH = (
    PROCESSED_DIR
    / "geopolitical_indices_daily_max.csv"
)

DAILY_SUM_PATH = (
    PROCESSED_DIR
    / "geopolitical_indices_daily_sum.csv"
)

DAILY_MA3_PATH = (
    PROCESSED_DIR
    / "geopolitical_indices_daily_sum_ma3.csv"
)

DAILY_MA5_PATH = (
    PROCESSED_DIR
    / "geopolitical_indices_daily_sum_ma5.csv"
)

DAILY_MA10_PATH = (
    PROCESSED_DIR
    / "geopolitical_indices_daily_sum_ma10.csv"
)

In [6]:
tweets = pd.read_csv(ANNOTATIONS_PATH)

print(f"Rows loaded: {len(tweets):,}")
print("\nColumns:")
print(tweets.columns.tolist())

print("\nFirst 5 rows:")
display(tweets.head())

print("\nMissing values:")
print(tweets.isna().sum())

Rows loaded: 15,278

Columns:
['tweet_index', 'tweet_id', 'date', 'tweet_snippet', 'trade_score', 'sanctions_score', 'fed_pressure_score', 'reasoning', 'source', 'example_ids', 'model', 'prompt_version', 'created_at_utc']

First 5 rows:


,tweet_index,tweet_id,date,tweet_snippet,trade_score,sanctions_score,fed_pressure_score,reasoning,source,example_ids,model,prompt_version,created_at_utc
0,0,8.154223e+17,2017-01-01 5:00,"TO ALL AMERICANS-#HappyNewYear &, many blessin...",0,0,0,The tweet is a New Year's greeting without ref...,llm:native_structured_output,A017|A030|A073|A036|A051|A035,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:14.544558+00:00
1,1,8.159307e+17,2017-01-02 14:40,"Well, the New Year begins. We will, together, ...",0,0,0,The tweet does not mention any Federal Reserve...,llm:native_structured_output,A040|A068|A078|A075|A022|A093,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:22.671079+00:00
2,2,8.159738e+17,2017-01-02 17:31,"Chicago murder rate is record setting - 4,331 ...",0,0,0,The tweet discusses crime statistics and reque...,llm:native_structured_output,A003|A092|A020|A058|A005|A096,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:31.742958+00:00
3,3,8.159892e+17,2017-01-02 18:32,"""@CNN just released a book called """"Unpreceden...",0,0,0,The tweet discusses a book release and critiqu...,llm:native_structured_output,A067|A069|A015|A095|A036|A058,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:40.130518+00:00
4,4,8.159903e+17,2017-01-02 18:37,Various media outlets and pundits say that I t...,0,0,0,The tweet discusses Trump's political victory ...,llm:native_structured_output,A039|A010|A100|A046|A076|A093,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:48.971179+00:00



Missing values:
tweet_index            0
tweet_id               0
date                   0
tweet_snippet          0
trade_score            0
sanctions_score        0
fed_pressure_score     0
reasoning              0
source                 2
example_ids           98
model                  2
prompt_version         2
created_at_utc         2
dtype: int64


In [7]:
# Columns used to construct the geopolitical indices
SCORE_COLS = [
    "trade_score",
    "sanctions_score",
    "fed_pressure_score",
]

# Convert date to datetime
tweets["date"] = pd.to_datetime(tweets["date"], errors="coerce")

# Check that dates converted successfully
assert tweets["date"].notna().all(), "Some dates could not be parsed."

# Check that scores are present and within the expected 0-100 range
assert tweets[SCORE_COLS].notna().all().all(), "Missing geopolitical scores found."

for col in SCORE_COLS:
    assert tweets[col].between(0, 100).all(), f"{col} contains values outside 0-100."

# Sort chronologically
tweets = tweets.sort_values("date").reset_index(drop=True)

print("Date range:")
print(tweets["date"].min(), "to", tweets["date"].max())

print("\nScore ranges:")
print(tweets[SCORE_COLS].agg(["min", "max"]))

print("\nTweets per day:")
print(tweets.groupby(tweets["date"].dt.date).size().describe())

Date range:
2017-01-01 05:00:00 to 2021-01-08 15:44:00

Score ranges:
     trade_score  sanctions_score  fed_pressure_score
min            0                0                   0
max           96               96                  99

Tweets per day:
count    1455.000000
mean       10.500344
std         7.560201
min         1.000000
25%         5.000000
50%         9.000000
75%        14.000000
max        61.000000
dtype: float64


## 1. Daily Aggregation of Tweet-Level Scores

Tweet-level geopolitical scores are aggregated by calendar day using two alternative measures.

The **daily maximum** records the highest score observed within each geopolitical category on a given day. This measure captures the intensity of the most geopolitically significant tweet posted that day.

The **daily sum** adds the scores of all tweets posted on each day. This measure captures both the intensity and frequency of geopolitical content and therefore allows multiple relevant tweets on the same day to contribute to the daily index.

In [8]:
# Create calendar-day variable while preserving original timestamp
tweets["calendar_date"] = tweets["date"].dt.normalize()

# Daily maximum score
daily_max = (
    tweets
    .groupby("calendar_date")[SCORE_COLS]
    .max()
    .reset_index()
)

daily_max = daily_max.rename(columns={
    "calendar_date": "date",
    "trade_score": "trade_max",
    "sanctions_score": "sanctions_max",
    "fed_pressure_score": "fed_pressure_max",
})

# Daily sum of scores
daily_sum = (
    tweets
    .groupby("calendar_date")[SCORE_COLS]
    .sum()
    .reset_index()
)

daily_sum = daily_sum.rename(columns={
    "calendar_date": "date",
    "trade_score": "trade_sum",
    "sanctions_score": "sanctions_sum",
    "fed_pressure_score": "fed_pressure_sum",
})

print("Tweeting days:", len(daily_sum))

print("\nDaily maximum:")
display(daily_max.head())

print("\nDaily sum:")
display(daily_sum.head())

Tweeting days: 1455

Daily maximum:


,date,trade_max,sanctions_max,fed_pressure_max
0,2017-01-01,0,0,0
1,2017-01-02,50,40,0
2,2017-01-03,50,0,0
3,2017-01-04,50,0,0
4,2017-01-05,40,0,0



Daily sum:


,date,trade_sum,sanctions_sum,fed_pressure_sum
0,2017-01-01,0,0,0
1,2017-01-02,50,60,0
2,2017-01-03,90,0,0
3,2017-01-04,50,0,0
4,2017-01-05,40,0,0


## 2. Construction of a Complete Daily Calendar

A complete calendar is constructed from the first to the final date in the tweet sample so that days on which no tweets were posted remain explicitly represented in the geopolitical time series.

For days without tweets, the geopolitical indices are assigned a value of zero. This represents the absence of a new tweet-based geopolitical signal rather than treating the observation as missing data.

This produces a continuous daily series suitable for alignment with the macroeconomic dataset.

In [9]:
# Complete calendar from first to last day in the sample
full_calendar = pd.DataFrame({
    "date": pd.date_range(
        start=tweets["calendar_date"].min(),
        end=tweets["calendar_date"].max(),
        freq="D"
    )
})

# Identify calendar days on which there were no tweets
no_tweet_days = full_calendar.loc[
    ~full_calendar["date"].isin(daily_sum["date"]),
    "date"
]

print(f"Total calendar days: {len(full_calendar)}")
print(f"Days with tweets: {len(daily_sum)}")
print(f"Days without tweets: {len(no_tweet_days)}")

print("\nDates with no tweets:")
print(no_tweet_days.to_string(index=False))

Total calendar days: 1469
Days with tweets: 1455
Days without tweets: 14

Dates with no tweets:
2017-03-12
2017-04-07
2017-04-15
2017-06-08
2017-08-13
2017-08-28
2018-01-30
2018-02-26
2018-05-06
2018-06-10
2018-09-28
2019-03-23
2020-08-09
2021-01-07


In [10]:
# Merge daily maximum onto complete calendar
daily_max = full_calendar.merge(
    daily_max,
    on="date",
    how="left"
)

# Merge daily sum onto complete calendar
daily_sum = full_calendar.merge(
    daily_sum,
    on="date",
    how="left"
)

# No tweet = no new geopolitical signal
daily_max[
    ["trade_max", "sanctions_max", "fed_pressure_max"]
] = daily_max[
    ["trade_max", "sanctions_max", "fed_pressure_max"]
].fillna(0)

daily_sum[
    ["trade_sum", "sanctions_sum", "fed_pressure_sum"]
] = daily_sum[
    ["trade_sum", "sanctions_sum", "fed_pressure_sum"]
].fillna(0)

print("Daily max rows:", len(daily_max))
print("Daily sum rows:", len(daily_sum))

print("\nMissing values in daily max:")
print(daily_max.isna().sum())

print("\nMissing values in daily sum:")
print(daily_sum.isna().sum())

print("\nExample no-tweet day:")
display(
    daily_sum[daily_sum["date"] == "2017-03-12"]
)

Daily max rows: 1469
Daily sum rows: 1469

Missing values in daily max:
date                0
trade_max           0
sanctions_max       0
fed_pressure_max    0
dtype: int64

Missing values in daily sum:
date                0
trade_sum           0
sanctions_sum       0
fed_pressure_sum    0
dtype: int64

Example no-tweet day:


,date,trade_sum,sanctions_sum,fed_pressure_sum
70,2017-03-12,0.0,0.0,0.0


## 3. Moving-Average Geopolitical Indices

To examine whether geopolitical information has a more persistent effect than a single-day measure can capture, moving averages of the daily summed indices are constructed over **3-, 5-, and 10-day windows**.

These measures smooth short-term fluctuations while allowing geopolitical signals to persist across subsequent days. Moving averages are calculated only when a complete rolling window is available, so the initial observations remain missing where insufficient historical data exists.

In [11]:
# Columns containing the daily summed geopolitical scores
SUM_COLS = [
    "trade_sum",
    "sanctions_sum",
    "fed_pressure_sum",
]

# Moving-average windows
windows = [3, 5, 10]

# Store each moving-average dataset
moving_averages = {}

for window in windows:
    ma = daily_sum[["date"]].copy()

    for col in SUM_COLS:
        ma[f"{col}_ma{window}"] = (
            daily_sum[col]
            .rolling(window=window, min_periods=window)
            .mean()
        )

    moving_averages[window] = ma

daily_ma3 = moving_averages[3]
daily_ma5 = moving_averages[5]
daily_ma10 = moving_averages[10]

print("MA(3):")
display(daily_ma3.head(12))

print("\nMA(5):")
display(daily_ma5.head(12))

print("\nMA(10):")
display(daily_ma10.head(12))

MA(3):


,date,trade_sum_ma3,sanctions_sum_ma3,fed_pressure_sum_ma3
0,2017-01-01,NaN,NaN,NaN
1,2017-01-02,NaN,NaN,NaN
2,2017-01-03,46.666667,20.000000,0.0
3,2017-01-04,63.333333,20.000000,0.0
4,2017-01-05,60.000000,0.000000,0.0
5,2017-01-06,43.333333,0.000000,0.0
6,2017-01-07,26.666667,0.000000,0.0
7,2017-01-08,13.333333,0.000000,0.0
8,2017-01-09,35.000000,8.333333,0.0
9,2017-01-10,35.000000,8.333333,0.0



MA(5):


,date,trade_sum_ma5,sanctions_sum_ma5,fed_pressure_sum_ma5
0,2017-01-01,NaN,NaN,NaN
1,2017-01-02,NaN,NaN,NaN
2,2017-01-03,NaN,NaN,NaN
3,2017-01-04,NaN,NaN,NaN
4,2017-01-05,46.0,12.0,0.0
5,2017-01-06,54.0,12.0,0.0
6,2017-01-07,44.0,0.0,0.0
7,2017-01-08,26.0,0.0,0.0
8,2017-01-09,37.0,5.0,0.0
9,2017-01-10,29.0,5.0,0.0



MA(10):


,date,trade_sum_ma10,sanctions_sum_ma10,fed_pressure_sum_ma10
0,2017-01-01,NaN,NaN,NaN
1,2017-01-02,NaN,NaN,NaN
2,2017-01-03,NaN,NaN,NaN
3,2017-01-04,NaN,NaN,NaN
4,2017-01-05,NaN,NaN,NaN
5,2017-01-06,NaN,NaN,NaN
6,2017-01-07,NaN,NaN,NaN
7,2017-01-08,NaN,NaN,NaN
8,2017-01-09,NaN,NaN,NaN
9,2017-01-10,37.5,8.5,0.0


## 4. Validation and Export

Before export, the constructed indices are validated to confirm the expected sample length, absence of unintended missing values, consistency between the maximum and summed measures, and the expected treatment of incomplete moving-average windows.

The five resulting geopolitical datasets are then exported for subsequent alignment with the macroeconomic variables and VAR-X modelling.

In [12]:
# Check expected number of rows
assert len(daily_max) == 1469
assert len(daily_sum) == 1469
assert len(daily_ma3) == 1469
assert len(daily_ma5) == 1469
assert len(daily_ma10) == 1469

# MAX and SUM should contain no missing values
assert daily_max.isna().sum().sum() == 0
assert daily_sum.isna().sum().sum() == 0

# SUM must always be at least as large as MAX
assert (daily_sum["trade_sum"] >= daily_max["trade_max"]).all()
assert (daily_sum["sanctions_sum"] >= daily_max["sanctions_max"]).all()
assert (daily_sum["fed_pressure_sum"] >= daily_max["fed_pressure_max"]).all()

# Moving averages should only have missing values
# where a complete rolling window does not yet exist
assert daily_ma3.iloc[:2, 1:].isna().all().all()
assert daily_ma3.iloc[2:, 1:].notna().all().all()

assert daily_ma5.iloc[:4, 1:].isna().all().all()
assert daily_ma5.iloc[4:, 1:].notna().all().all()

assert daily_ma10.iloc[:9, 1:].isna().all().all()
assert daily_ma10.iloc[9:, 1:].notna().all().all()

print("All validation checks passed.")

print("\nDataset dimensions:")
print("Daily MAX:   ", daily_max.shape)
print("Daily SUM:   ", daily_sum.shape)
print("SUM MA(3):   ", daily_ma3.shape)
print("SUM MA(5):   ", daily_ma5.shape)
print("SUM MA(10):  ", daily_ma10.shape)

All validation checks passed.

Dataset dimensions:
Daily MAX:    (1469, 4)
Daily SUM:    (1469, 4)
SUM MA(3):    (1469, 4)
SUM MA(5):    (1469, 4)
SUM MA(10):   (1469, 4)


In [13]:
# Save daily maximum and daily sum indices
daily_max.to_csv(DAILY_MAX_PATH, index=False)
daily_sum.to_csv(DAILY_SUM_PATH, index=False)

# Save moving-average versions of the daily sum
daily_ma3.to_csv(DAILY_MA3_PATH, index=False)
daily_ma5.to_csv(DAILY_MA5_PATH, index=False)
daily_ma10.to_csv(DAILY_MA10_PATH, index=False)

print("Saved:")
print(DAILY_MAX_PATH)
print(DAILY_SUM_PATH)
print(DAILY_MA3_PATH)
print(DAILY_MA5_PATH)
print(DAILY_MA10_PATH)

Saved:
..\data\processed\geopolitical_indices_daily_max.csv
..\data\processed\geopolitical_indices_daily_sum.csv
..\data\processed\geopolitical_indices_daily_sum_ma3.csv
..\data\processed\geopolitical_indices_daily_sum_ma5.csv
..\data\processed\geopolitical_indices_daily_sum_ma10.csv


In [14]:
saved_files = {
    "daily_max": DAILY_MAX_PATH,
    "daily_sum": DAILY_SUM_PATH,
    "daily_ma3": DAILY_MA3_PATH,
    "daily_ma5": DAILY_MA5_PATH,
    "daily_ma10": DAILY_MA10_PATH,
}

for name, path in saved_files.items():
    df = pd.read_csv(path)

    print(
        f"{name}: "
        f"{df.shape[0]} rows, "
        f"{df.shape[1]} columns"
    )

daily_max: 1469 rows, 4 columns
daily_sum: 1469 rows, 4 columns
daily_ma3: 1469 rows, 4 columns
daily_ma5: 1469 rows, 4 columns
daily_ma10: 1469 rows, 4 columns


## 5. Summary

Tweet-level LLM geopolitical-risk scores were transformed into five alternative daily representations for each of the Trade Hostility, Sanctions Threat and Federal Reserve Pressure indices.

The resulting datasets comprise a daily maximum measure, a daily summed measure, and 3-, 5-, and 10-day moving averages of the summed scores. A complete calendar was retained throughout the sample, with no-tweet days assigned zero geopolitical signal.

These alternative specifications allow the subsequent modelling analysis to assess whether EUR/USD dynamics are better explained by the most intense daily geopolitical event, the cumulative amount of geopolitical content, or a smoothed measure capturing persistence across several days.